# Nível 1 — Análise de Operações e PLD

Este notebook implementa o tratamento dos dados, as regras determinísticas de sinalização e a análise de um cliente sinalizado utilizando um modelo de linguagem.

A solução separa os cálculos determinísticos, realizados com pandas, da interpretação qualitativa realizada pelo LLM.

In [65]:
import json
import time
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

In [66]:
CAMINHO_DADOS = Path("../dados/dados_nivel_1.json")

with open(CAMINHO_DADOS, "r", encoding="utf-8") as arquivo:
    dados = json.load(arquivo)

taxa_cambio = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])

print(f"Taxa USD/BRL: {taxa_cambio}")
print(f"Quantidade inicial de registros: {len(df)}")

display(df.head())

Taxa USD/BRL: 5.4
Quantidade inicial de registros: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [67]:
df.info()

print("\nValores ausentes:")
display(df.isna().sum().to_frame("quantidade"))

print("\nIDs duplicados:")
display(
    df[df.duplicated(subset=["id"], keep=False)]
    .sort_values("id")
)

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB

Valores ausentes:


,quantidade
id,0
cliente_id,0
data,1
valor,0
moeda,0
canal,0
tipo,0
contraparte,0
observacao,0



IDs duplicados:


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [68]:
print("Quantidade de clientes:", df["cliente_id"].nunique())
print("Quantidade de IDs únicos:", df["id"].nunique())
print("Quantidade de registros:", len(df))

print("\nMoedas:")
display(df["moeda"].value_counts())

print("\nCanais:")
display(df["canal"].value_counts())

print("\nTipos:")
display(df["tipo"].value_counts())

Quantidade de clientes: 6
Quantidade de IDs únicos: 19
Quantidade de registros: 20

Moedas:


moeda
BRL    19
USD     1
Name: count, dtype: int64


Canais:


canal
pix        9
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64


Tipos:


tipo
transferencia_enviada     11
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

## Tratamento dos dados

Foram identificados dois problemas de qualidade:

1. A operação `OP-0007` aparece duplicada com os mesmos atributos. Foi mantida apenas uma ocorrência para evitar dupla contagem e impacto indevido nas regras determinísticas.
2. A operação `OP-0017` possui data ausente. A operação foi mantida, pois ainda é válida para análises que não dependem de data, mas a ausência foi representada como `NaT` e essa operação não participa de análises que exigem agrupamento temporal.

Além disso, os valores foram normalizados para BRL utilizando a taxa de câmbio fixa fornecida no próprio arquivo.

In [69]:
df_limpo = df.copy()

# Remove duplicidades pelo identificador da operação
df_limpo = df_limpo.drop_duplicates(subset=["id"], keep="first").copy()

# Converte a coluna de data
df_limpo["data"] = pd.to_datetime(df_limpo["data"], errors="coerce")

print("Registros antes da limpeza:", len(df))
print("Registros após a limpeza:", len(df_limpo))
print("Datas ausentes após conversão:", df_limpo["data"].isna().sum())

Registros antes da limpeza: 20
Registros após a limpeza: 19
Datas ausentes após conversão: 1


In [70]:
df_limpo["valor_brl"] = df_limpo.apply(
    lambda linha: linha["valor"] * taxa_cambio
    if linha["moeda"] == "USD"
    else linha["valor"],
    axis=1
)

display(
    df_limpo[["id", "cliente_id", "valor", "moeda", "valor_brl"]]
)

,id,cliente_id,valor,moeda,valor_brl
0,OP-0001,CLI-A-1,18100,BRL,18100.0
1,OP-0002,CLI-A-1,17300,BRL,17300.0
2,OP-0003,CLI-A-1,18800,BRL,18800.0
3,OP-0004,CLI-A-1,3300,BRL,3300.0
4,OP-0005,CLI-A-2,25900,BRL,25900.0
5,OP-0006,CLI-A-2,27000,BRL,27000.0
6,OP-0007,CLI-A-3,17200,BRL,17200.0
7,OP-0008,CLI-A-3,15200,BRL,15200.0
8,OP-0009,CLI-A-3,16100,BRL,16100.0
10,OP-0010,CLI-A-4,3800,BRL,3800.0


In [71]:
volume_por_cliente = (
    df_limpo.groupby("cliente_id", as_index=False)["valor_brl"]
    .sum()
    .rename(columns={"valor_brl": "volume_total_brl"})
    .sort_values("volume_total_brl", ascending=False)
)

print("Volume total transacionado por cliente:")
display(volume_por_cliente)

Volume total transacionado por cliente:


,cliente_id,volume_total_brl
3,CLI-A-4,79500.0
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


In [72]:
operacoes_por_canal = (
    df_limpo.groupby("canal")
    .size()
    .reset_index(name="quantidade_operacoes")
    .sort_values("quantidade_operacoes", ascending=False)
)

print("Quantidade de operações por canal:")
display(operacoes_por_canal)

Quantidade de operações por canal:


,canal,quantidade_operacoes
3,pix,8
4,ted,5
0,boleto,3
1,cartao,2
2,especie,1


## Regras determinísticas

As regras abaixo são calculadas exclusivamente com pandas. O modelo de linguagem será utilizado apenas posteriormente para interpretar os casos sinalizados.

In [73]:
# Inicializa a flag como False
df_limpo["flag_fracionamento"] = False

# Considera apenas operações com data conhecida
df_com_data = df_limpo.dropna(subset=["data"]).copy()

# Agrega por cliente e data
resumo_dia = (
    df_com_data
    .groupby(["cliente_id", "data"])
    .agg(
        quantidade_operacoes=("id", "count"),
        soma_dia_brl=("valor_brl", "sum"),
        maior_operacao_brl=("valor_brl", "max")
    )
    .reset_index()
)

# Aplica a regra
casos_fracionamento = resumo_dia[
    (resumo_dia["quantidade_operacoes"] >= 3)
    & (resumo_dia["soma_dia_brl"] > 50000)
    & (resumo_dia["maior_operacao_brl"] < 20000)
].copy()

display(casos_fracionamento)

,cliente_id,data,quantidade_operacoes,soma_dia_brl,maior_operacao_brl
0,CLI-A-1,2026-03-09,3,54200.0,18800.0


In [74]:
for _, caso in casos_fracionamento.iterrows():
    mascara = (
        (df_limpo["cliente_id"] == caso["cliente_id"])
        & (df_limpo["data"] == caso["data"])
    )
    df_limpo.loc[mascara, "flag_fracionamento"] = True

display(
    df_limpo[df_limpo["flag_fracionamento"]][
        ["id", "cliente_id", "data", "valor_brl", "flag_fracionamento"]
    ]
)

,id,cliente_id,data,valor_brl,flag_fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100.0,True
1,OP-0002,CLI-A-1,2026-03-09,17300.0,True
2,OP-0003,CLI-A-1,2026-03-09,18800.0,True


In [75]:
validacao_regra1 = resumo_dia[
    (
        (resumo_dia["cliente_id"] == "CLI-A-1")
        & (resumo_dia["data"] == pd.Timestamp("2026-03-09"))
    )
    |
    (
        (resumo_dia["cliente_id"] == "CLI-A-3")
        & (resumo_dia["data"] == pd.Timestamp("2026-03-05"))
    )
].copy()

display(validacao_regra1)

,cliente_id,data,quantidade_operacoes,soma_dia_brl,maior_operacao_brl
0,CLI-A-1,2026-03-09,3,54200.0,18800.0
3,CLI-A-3,2026-03-05,3,48500.0,17200.0


### Validação da Regra 1

O cliente `CLI-A-1` foi corretamente sinalizado, pois realizou 3 operações na mesma data, totalizando mais de R$ 50.000, sem que nenhuma operação isolada atingisse R$ 20.000.

O cliente `CLI-A-3` apresenta um padrão semelhante de 3 operações na mesma data, porém a soma permanece abaixo de R$ 50.000. Por isso, não é sinalizado pela regra.

Essa comparação também demonstra a importância da remoção da operação duplicada identificada durante a limpeza.

In [76]:
# Quantidade de operações e mediana por cliente
estatisticas_cliente = (
    df_limpo
    .groupby("cliente_id")
    .agg(
        quantidade_operacoes=("id", "count"),
        mediana_valor_brl=("valor_brl", "median")
    )
    .reset_index()
)

display(estatisticas_cliente)

,cliente_id,quantidade_operacoes,mediana_valor_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


In [77]:
df_limpo = df_limpo.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left"
)

df_limpo["flag_valor_atipico"] = (
    (df_limpo["quantidade_operacoes"] >= 4)
    & (df_limpo["valor_brl"] > 5 * df_limpo["mediana_valor_brl"])
)

display(
    df_limpo[df_limpo["flag_valor_atipico"]][
        [
            "id",
            "cliente_id",
            "valor_brl",
            "mediana_valor_brl",
            "quantidade_operacoes",
            "flag_valor_atipico"
        ]
    ]
)

,id,cliente_id,valor_brl,mediana_valor_brl,quantidade_operacoes,flag_valor_atipico
12,OP-0013,CLI-A-4,64800.0,5450.0,4,True


## Parte B — Análise qualitativa com LLM

Para a análise qualitativa foi selecionado o cliente `CLI-A-4`, sinalizado pela Regra 2 (valor atípico).

Os cálculos de quantidade de operações, mediana, conversão cambial e comparação com o limite foram realizados previamente com pandas. O modelo de linguagem recebe esses resultados como fatos e é utilizado somente para interpretação e redação do parecer.

In [78]:
cliente_escolhido = "CLI-A-4"

operacoes_cliente = df_limpo[
    df_limpo["cliente_id"] == cliente_escolhido
].copy()

display(
    operacoes_cliente[
        [
            "id",
            "data",
            "valor",
            "moeda",
            "valor_brl",
            "canal",
            "tipo",
            "contraparte",
            "flag_valor_atipico"
        ]
    ]
)

,id,data,valor,moeda,valor_brl,canal,tipo,contraparte,flag_valor_atipico
9,OP-0010,2026-03-03,3800,BRL,3800.0,cartao,pagamento,Alfa Comercio LTDA,False
10,OP-0011,2026-03-11,5100,BRL,5100.0,boleto,pagamento,Beta Servicos ME,False
11,OP-0012,2026-03-18,5800,BRL,5800.0,pix,transferencia_enviada,Gama Distribuidora,False
12,OP-0013,2026-03-24,12000,USD,64800.0,ted,transferencia_recebida,Zeta Importacao,True


In [79]:
estatistica_cliente = estatisticas_cliente[
    estatisticas_cliente["cliente_id"] == cliente_escolhido
].iloc[0]

contexto_cliente = {
    "cliente_id": cliente_escolhido,
    "quantidade_operacoes": int(estatistica_cliente["quantidade_operacoes"]),
    "mediana_valor_brl": float(estatistica_cliente["mediana_valor_brl"]),
    "operacoes": operacoes_cliente[
        [
            "id",
            "data",
            "valor_brl",
            "canal",
            "tipo",
            "contraparte",
            "flag_fracionamento",
            "flag_valor_atipico"
        ]
    ].assign(
        data=lambda x: x["data"].astype(str)
    ).to_dict(orient="records")
}

print(json.dumps(contexto_cliente, indent=2, ensure_ascii=False))

{
  "cliente_id": "CLI-A-4",
  "quantidade_operacoes": 4,
  "mediana_valor_brl": 5450.0,
  "operacoes": [
    {
      "id": "OP-0010",
      "data": "2026-03-03",
      "valor_brl": 3800.0,
      "canal": "cartao",
      "tipo": "pagamento",
      "contraparte": "Alfa Comercio LTDA",
      "flag_fracionamento": false,
      "flag_valor_atipico": false
    },
    {
      "id": "OP-0011",
      "data": "2026-03-11",
      "valor_brl": 5100.0,
      "canal": "boleto",
      "tipo": "pagamento",
      "contraparte": "Beta Servicos ME",
      "flag_fracionamento": false,
      "flag_valor_atipico": false
    },
    {
      "id": "OP-0012",
      "data": "2026-03-18",
      "valor_brl": 5800.0,
      "canal": "pix",
      "tipo": "transferencia_enviada",
      "contraparte": "Gama Distribuidora",
      "flag_fracionamento": false,
      "flag_valor_atipico": false
    },
    {
      "id": "OP-0013",
      "data": "2026-03-24",
      "valor_brl": 64800.00000000001,
      "canal": "ted",
     

In [80]:
from typing import Literal
from pydantic import BaseModel, ValidationError


class ParecerPLD(BaseModel):
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

In [82]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY não encontrada no arquivo .env")

client = Groq(api_key=api_key)